# تحميل القصص/الفيديوهات → Drive
1) شغّل الخلية الأولى (تسجيل الدخول Google + تهيئة Drive).
2) ضع الرابط وعدد العناصر في الخلية الثانية وشغّلها.
3) شغّل الخلية الثالثة: إزالة شعار المنصة فقط + رفع مرقّم إلى Drive.

ملاحظة: الترقيم حسب ترتيب ما يُرجعه Instagram/تيك توك (الأحدث غالبًا أولًا).

In [ ]:
!pip -q install --upgrade 'yt-dlp[default]' imageio-ffmpeg
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
import imageio_ffmpeg, os
os.environ['IMAGEIO_FFMPEG_EXE'] = imageio_ffmpeg.get_ffmpeg_exe()
drive = build('drive', 'v3')
print('Google Drive جاهز ✅')

In [ ]:
import yt_dlp, shutil
from pathlib import Path
from urllib.parse import urlparse

URL = 'https://www.instagram.com/USERNAME/'  # @param {type:"string"}
LIMIT = 200  # @param {type:"integer"}
FOLDER_NAME = ''  # @param {type:"string"}

OUT = Path('/content/stories')
shutil.rmtree(OUT, ignore_errors=True)
OUT.mkdir(parents=True, exist_ok=True)

opts = {
    'outtmpl': str(OUT / '%(playlist_index)03d-%(id)s.%(ext)s'),
    'playlistend': LIMIT,
    'noplaylist': False,
    'ignoreerrors': True,
    'retries': 5,
    'socket_timeout': 30,
    'merge_output_format': 'mp4',
    'ffmpeg_location': imageio_ffmpeg.get_ffmpeg_exe(),
    'writethumbnail': False,
    'writeinfojson': False,
    'quiet': False,
    'no_warnings': False,
}
with yt_dlp.YoutubeDL(opts) as ydl:
    info = ydl.extract_info(URL, download=True)

host = (urlparse(URL).hostname or '').lower()
parts = [p for p in urlparse(URL).path.split('/') if p]
if 'instagram' in host:
    owner = parts[1] if parts and parts[0].lower() == 'stories' and len(parts) > 1 else (parts[0] if parts else '')
elif 'tiktok' in host:
    owner = next((p.lstrip('@') for p in parts if p.startswith('@')), '')
else:
    owner = parts[0] if parts else ''
if not owner and isinstance(info, dict):
    owner = info.get('channel') or info.get('uploader') or info.get('uploader_id') or ''
FOLDER_NAME = (FOLDER_NAME or str(owner) or 'stories').strip()

files = sorted([p for p in OUT.iterdir() if p.is_file()], key=lambda p: p.name)
print('عدد الملفات:', len(files))
print('اسم المجلد:', FOLDER_NAME)
print([p.name for p in files[:10]])

In [ ]:
import subprocess, re

REGIONS = {
    'instagram': 'W-130:20:120:80',
    'tiktok': 'W-140:H-110:130:80',
    'facebook': 'W-140:H-100:130:80',
}
PLATFORM = 'instagram' if 'instagram' in host else ('tiktok' if 'tiktok' in host else ('facebook' if 'facebook' in host else 'generic'))
FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()

def _value(token, w, h):
    m = re.fullmatch(r'([WH])?\s*([+-]?\d+)', (token or '').strip().upper())
    if not m:
        raise ValueError(token)
    base = w if m.group(1) == 'W' else (h if m.group(1) == 'H' else 0)
    return base + int(m.group(2))

def region(spec, w, h):
    try:
        x, y, rw, rh = [_value(t, w, h) for t in spec.split(':')]
    except Exception:
        return None
    if w <= 0 or h <= 0 or rw <= 0 or rh <= 0:
        return None
    x = max(0, min(w - 2, x)); y = max(0, min(h - 2, y))
    rw = min(rw, w - x); rh = min(rh, h - y)
    x, y, rw, rh = x - x % 2, y - y % 2, rw - rw % 2, rh - rh % 2
    return (x, y, rw, rh) if rw >= 2 and rh >= 2 else None

def probe_size(path):
    out = subprocess.run([FFMPEG, '-hide_banner', '-i', str(path)], capture_output=True, text=True).stderr
    m = re.search(r'(\d{2,5})x(\d{2,5})', out)
    return (int(m.group(1)), int(m.group(2))) if m else (0, 0)

def remove_logo(src):
    w, h = probe_size(src)
    r = region(REGIONS.get(PLATFORM, ''), w, h)
    if r is None:
        print('⚠️ منطقة غير صالحة، إبقاء الأصل:', src.name)
        return src, False
    x, y, rw, rh = r
    dst = src.with_name(src.stem + '.clean.mp4')
    cmd = [FFMPEG, '-hide_banner', '-loglevel', 'error', '-y', '-threads', '1',
           '-i', str(src), '-map', '0:v:0', '-map', '0:a?',
           '-vf', f'delogo=x={x}:y={y}:w={rw}:h={rh}',
           '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '24',
           '-pix_fmt', 'yuv420p', '-c:a', 'aac', '-b:a', '96k', str(dst)]
    done = subprocess.run(cmd, capture_output=True, text=True)
    if done.returncode != 0 or not dst.exists() or dst.stat().st_size == 0:
        dst.unlink(missing_ok=True)
        print('⚠️ فشل delogo، إبقاء الأصل:', src.name, done.stderr[-200:])
        return src, False
    return dst, True

folder = drive.files().create(body={
    'name': FOLDER_NAME,
    'mimeType': 'application/vnd.google-apps.folder',
}, fields='id,webViewLink').execute()
folder_id = folder['id']

for index, src in enumerate(files, 1):
    cleaned, changed = remove_logo(src) if src.suffix.lower() in {'.mp4', '.mkv', '.webm', '.mov', '.m4v'} else (src, False)
    ext = cleaned.suffix.lower()
    if ext not in {'.mp4', '.jpg', '.png', '.webp'}:
        ext = '.mp4'
    name = f'{index:03d}{ext}'
    mime = {'.jpg': 'image/jpeg', '.png': 'image/png', '.webp': 'image/webp'}.get(ext, 'video/mp4')
    media = MediaFileUpload(str(cleaned), mimetype=mime, resumable=True)
    created = drive.files().create(
        body={'name': name, 'parents': [folder_id]},
        media_body=media, fields='id,webViewLink',
    ).execute()
    print(f'{index:03d}', src.name, '->', name, 'شعار مزال' if changed else 'كما هو', created.get('webViewLink', ''))

print('📁 المجلد:', f'https://drive.google.com/drive/folders/{folder_id}')